# Imports

In [449]:
import pandas as pd
import io
import re
import time
from tqdm import tqdm
import tiktoken
import google.generativeai as genai
from google.generativeai.types import GenerationConfig
import os

# Configuration

In [450]:
# List all models available for content generation
print("Available models:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f" - {m.name}")

Available models:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.0-flash
 - models/gemini-2.0-flash-001
 - models/gemini-2.0-flash-lite-001
 - models/gemini-2.0-flash-lite
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-pro-preview
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/lyria-3-clip-preview
 - models/lyria-3-pro-preview
 - models/gemini-3.1-flash-tts-preview
 - models/gemini-robotics-er-1.5-preview
 - models/gemini-robotics-er-1.6-preview
 - model

In [456]:
# ==========================================
# CONFIGURATION
# ==========================================

API_KEY    = "place your api"
MODEL      = "models/gemini-3.1-flash-lite"
INPUT_CSV  = "/kaggle/input/datasets/mahmoudmohamedfawzy/mental-health-detection-nlp/mental_health.csv"
OUTPUT_CSV = "/kaggle/working//gemini_labels_output.csv"

In [452]:
RPM_LIMIT   = 28
WINDOW_SEC  = 60     # 60 Sec to handle 1 request per min
REST_SEC    = 35
# --- NEW CONFIGURATIONS FOR DYNAMIC CHUNKING ---
TPM_LIMIT = 250000   # e.g., 1 Million Tokens Per Minute for Gemini 2.5 Flash
SAFE_ZONE_PERCENT = 0.75  # 20% buffer (keeps chunk size to max 80% of TPM limit)
# 2. OUTPUT LIMIT: Dynamically detect model max output tokens
model_name = MODEL.lower()
if "2.5" in model_name or "3.1" in model_name:
    # Gemini 2.5 and 3.x models support up to 65,536 output tokens
    MAX_TOKENS = 65536 # e.g., 65K Max Output Tokens Per request for Gemini 2.5 Flash
else:
    # Gemini 1.5 / 2.0 / TTS models hard cap at 8,192 output tokens
    MAX_TOKENS = 8192 # e.g., 65K Max Output Tokens Per request for Gemini 2.5 Flash

# Few Shot Prompt + LLM For Labeling

In [454]:
# ==========================================
# PROMPT DEFINITION
# ==========================================

SUMMARY_PROMPT = """\
# Instruction
You are a clinical NLP expert and mental health classifier. Your task is to process a batch of text messages provided as a CSV chunk and assess each person's mental state on a scale of 1 to 10.

You will receive the input data as a CSV string with the following columns: 
row_id, text, label

You MUST return the processed data as a raw CSV string with these exact columns in this exact order:
row_id, reasoning, dominant_signals, score, rating

Here is the brief description about the dataset:
To classify if the person is struggling with mental health issues or not from their messages.
Label with 0 means mentally well and score will usually be less than or equal to 5. Label with 1 means not mentally well, which means score would be greater than 5.
This dataset is a collection of texts related to people with anxiety, depression, and other mental health issues. The corpus consists of two columns: one containing the comments, and the other containing labels indicating whether the comments are considered indicative of mental health issues or not. The data may be useful for researchers, mental health professionals, and others interested in understanding the language and sentiment surrounding mental health issues. 1 means considered as a comment which is indicative of mental health issues, and 0 means a person who is not struggling with mental health issues.

Understanding and detecting mental health issues through text messages can be a critical step in providing timely support and intervention for those in need. Research has shown that linguistic patterns and word choices in written communication can be indicative of various mental health conditions, including depression, anxiety, and stress. Analyzing the content of messages, along with the intensity of emotions conveyed, can offer valuable insights into a person's emotional well-being.

This cutting-edge field combines NLP techniques with psychology and psychiatry, aiming to build automated systems capable of identifying signs of mental distress accurately.

You should evaluate each row based on the Criteria and Rating Rubric below. Give step-by-step reasoning for your rating.

---

# Evaluation

## Metric Definition
You will be assessing the mental health state of a person based on their text. The response must be grounded only in the provided text — do not infer or assume information not present.

## Criteria
- **Mental Wellbeing**: Overall emotional and psychological state of the person.
- **Hopelessness**: Presence of nihilistic thinking, loss of purpose, or belief that things cannot improve.
- **Anxiety**: Signs of excessive worry, fear, panic, or rumination.
- **Depression**: Signs of persistent sadness, withdrawal, low energy, or emotional numbness.
- **Stress**: Indicators of being overwhelmed, burned out, or under pressure.
- **Suicidal Ideation**: Any direct or indirect references to self-harm or not wanting to live.
- **Groundedness**: Assessment must be based only on what the text says; do not hallucinate signals.

## Rating Rubric
- 1 (BEST): Excellent mental state. No signs of distress or mental health concerns whatsoever.
- 2 (VERY_GOOD): Mentally well with very minor or no issues; no immediate concerns.
- 3 (GOOD): Healthy mental state with minor or occasional concerns; generally doing well.
- 4 (MODERATE_GOOD): Fairly good mental state; minimal issues present.
- 5 (NEUTRAL): Borderline — neither clearly healthy nor unhealthy; some ambiguous signals.
- 6 (MODERATE_BAD): Moderate signs of mental health issues; monitoring or professional support advised.
- 7 (BAD): Significantly compromised mental state; recommend seeking help soon.
- 8 (VERY_BAD): Severely compromised; professional assistance needed promptly.
- 9 (VERY_VERY_BAD): Very severely compromised; immediate intervention strongly advised.
- 10 (WORST): Extremely poor mental state; urgent crisis intervention required.

## Evaluation Steps (Apply to EACH row)
- STEP 1: Read the text carefully and identify linguistic signals related to mental health.
- STEP 2: Cross-reference identified signals against the Criteria.
- STEP 3: Assign a score strictly following the Rating Rubric, ensuring consistency with the provided label (label 0 → score ≤ 5, label 1 → score > 5).
- STEP 4: List the dominant signals as a comma-separated string (e.g., "social isolation, hopelessness").
- STEP 5: Write a concise 2–3 sentence reasoning explaining your score.

---

# Few-Shot Examples (Format Reference)

**Input CSV (What you receive):**
row_id,text,label
100,"Had the most amazing weekend hiking with friends. Feeling so refreshed and grateful for life. Can't wait for next weekend!",0
101,"guys finally got a girlfriend after leaving a toxic relationship that had a negative effect on my wellbeing. got help writing a text, my dad said he was proud of how i dealt with it. could barely believe it. nice to see there are loads of posts about people getting into good relationships",0
102,"Work has been a bit stressful lately but I've been managing it fine. Going to the gym helps a lot. My friends have been super supportive too.",0
103,"tell my crush i like her ive been procrastinating for months at this point im still unsure about it get help pls",0
104,"I don't really know how I'm feeling these days. Some days are okay, some days just feel really heavy. I'm managing but it's not easy.",0
105,"world ppl cares give them i planet yrs one thing learned ppl care u something give them im tired wish born way care productive wanna connect ppl want cant connect anyone awful feel trapped",1
106,"i've been crying every day for weeks and i don't even know why. i can't get out of bed most mornings. i've stopped replying to my friends. everything just feels pointless and heavy.",1
107,"dont know how long i can hold on longer. thought my boyfriend could be enough to help hold me. possibly going to start meds even though i hate anything like that. theres nothing going anywhere in my life. its fucking meaningless. cant continue anymore.",1
108,"nothing to look forward to in life. dont have many reasons to keep going. feel like nothing keeps me going to the next day. makes me want to hang.",1
109,"cant do this anymore. tried to kill myself twice. wish id succeeded last summer, a few months ago. was hospitalized. couldnt do anything. cant say it really helped. told my pdoc and therapist as well two weeks ago. not sure about telling people anymore.",1

**Output CSV (What you MUST return based on the Input CSV):**
row_id,reasoning,dominant_signals,score,rating
100,"Expresses joy, gratitude, and social connection. No indicators of distress, anxiety, or depression. Person is thriving.","positive affect, social engagement, forward-looking mindset",1,BEST
101,"Positive and forward-looking narrative. Healthy coping and growth after recovering from a difficult relationship. Strong support network present.","recovery, positive affect, social support, healthy growth",2,VERY_GOOD
102,"Acknowledges some stress but demonstrates active and healthy coping mechanisms. Strong social and physical health buffers are in place.","mild work stress, healthy coping, social support, physical activity",3,GOOD
103,"Person is nervous about a normal social situation. Shows some anxiety around interpersonal interaction but nothing indicative of a mental health disorder. Seeking lighthearted advice.","social nervousness, mild indecision, no clinical distress signals",4,MODERATE_GOOD
104,"Ambiguous emotional state — neither clearly distressed nor clearly healthy. The person is coping but borderline; emotional heaviness is noted without acute crisis signals.","emotional ambiguity, fluctuating mood, mild low affect, no acute crisis",5,NEUTRAL
105,"Deep social isolation, feeling trapped, exhaustion, and inability to connect with others are present. Multiple moderate depression markers including emotional withdrawal and hopelessness are evident.","social isolation, feeling trapped, hopelessness, emotional withdrawal, fatigue",6,MODERATE_BAD
106,"Persistent depressive symptoms including anhedonia, social withdrawal, unexplained crying, and loss of motivation are all present. Functioning is clearly impaired and professional help is recommended.","persistent crying, social withdrawal, anhedonia, impaired functioning, low motivation",7,BAD
107,"Clear hopelessness, perceived meaninglessness, and strong implicit suicidal ideation are present. Person feels unsupported and is struggling severely with daily functioning.","hopelessness, meaninglessness, implicit suicidal ideation, emotional exhaustion, loss of will",8,VERY_BAD
108,"Directly expresses suicidal ideation and a complete absence of hope or reason to live. Crisis-level message requiring immediate intervention.","explicit suicidal ideation, total hopelessness, no future orientation, desire to die",9,VERY_VERY_BAD
109,"Two prior suicide attempts with expressed regret at survival, recent hospitalization, and eroding trust in professional help. This is an extreme, immediate crisis situation.","multiple suicide attempts, survivor's regret, loss of trust in professionals, active crisis, isolation from support",10,WORST

---

# CRITICAL RULES FOR YOUR OUTPUT:
1. Return ONLY valid CSV data — no markdown code fences (```), no explanations, no preamble, and no extra text.
2. The output MUST contain the exact same number of rows as the input.
3. Preserve the row_id values exactly.
4. Use standard CSV format with comma separators. You MUST enclose the reasoning and dominant_signals strings in double quotes ("") since they contain commas.
5. The output columns must be in this exact order: row_id, reasoning, dominant_signals, score, rating.

---

# User Input (CSV Data to Process)

{text}
"""

def build_prompt(csv_text: str) -> str:
    return SUMMARY_PROMPT.format(text=csv_text)
    
# ==========================================
# HELPER FUNCTIONS
# ==========================================
def extract_csv(raw: str, expected_rows: int = None) -> pd.DataFrame:
    """Strip think-blocks / fences, then parse CSV response."""
    raw = re.sub(r"<tool_call>.*?</tool_call>", "", raw, flags=re.DOTALL)
    raw = re.sub(r"http://googleusercontent.com/immersive_entry_chip/0", "", raw)
    
    # FIX 1: Safely remove ```csv, ```json, or standard ```
    raw = re.sub(r"```[a-zA-Z]*\n?", "", raw).strip()

    lines = raw.split("\n")
    csv_lines = []
    in_csv = False
    
    for line in lines:
        stripped = line.strip()
        if not in_csv:
            if "row_id" in stripped and ("score" in stripped or "reasoning" in stripped):
                in_csv = True
                csv_lines.append(line)
        else:
            # End of block if model closes with backticks
            if stripped.startswith("```"):
                break
            # FIX 2: Don't break on empty strings, just skip them to avoid cutting the CSV short
            if stripped != "":
                csv_lines.append(line)

    if not csv_lines: 
        csv_lines = lines
        
    raw_csv = "\n".join(csv_lines)

    try:
        df = pd.read_csv(io.StringIO(raw_csv))
        
        # FIX 3: Strip whitespace from column names (critical for subset check)
        df.columns = df.columns.str.strip()
        
        if expected_rows and len(df) != expected_rows:
            print(f"  Warning: expected {expected_rows} rows, got {len(df)}")
        
        required = {"row_id", "reasoning", "dominant_signals", "score", "rating"}
        if not required.issubset(set(df.columns)):
            print(f"  Warning: Missing columns. Found: {list(df.columns)}")
            return None
            
        df["score"] = pd.to_numeric(df["score"], errors="coerce")
        df["row_id"] = pd.to_numeric(df["row_id"], errors="coerce")
        return df
        
    except Exception as e:
        print(f"  CSV Parse Error: {e}")
        return None

# ==========================================
# TOKEN & CHUNK CALCULATIONS
# ==========================================
def estimate_tokens(text: str) -> int:
    """Estimates the number of tokens in a given text string."""
    # Heuristic: 1 token is approximately 4 characters
    return max(1, len(str(text)) // 4)

def calculate_dynamic_chunk_size(df: pd.DataFrame, tpm_limit: int, safe_zone_pct: float) -> int:
    """Calculates the max rows per chunk based on both Input TPM and Model Output Limits."""
    base_prompt_tokens = 2500  # Estimate for system prompt + instructions + examples
    
    # Estimate average input tokens per row
    sample_size = min(len(df), 1000)
    sample_df = df.head(sample_size).copy()
    sample_df['estimated_tokens'] = sample_df['text'].apply(estimate_tokens)
    avg_input_tokens = sample_df['estimated_tokens'].mean()
    
    # 1. INPUT LIMIT: How many rows fit in the TPM limit?
    available_input = (tpm_limit * (1.0 - safe_zone_pct)) - base_prompt_tokens
    max_rows_by_input = int(available_input // avg_input_tokens)
            
    # Each row of output (reasoning, signals, score, rating) takes ~80 tokens on average
    estimated_out_per_row = 80
    max_output_limit = MAX_TOKENS * (1.0 - safe_zone_pct) # Apply safe zone buffer
    max_rows_by_output = int(max_output_limit // estimated_out_per_row)
    
    # The theoretical chunk size
    theoretical_chunk = min(max_rows_by_input, max_rows_by_output)
    
    
    dynamic_chunk_size = theoretical_chunk #max(1, min(theoretical_chunk, 25)) # HARD CAP: Never exceed 25 rows to prevent LLM formatting degradation/truncation
    
    print(f"--- [DEBUG] Dynamic Chunk Configuration ---")
    print(f"Model Selected: {MODEL}")
    print(f"Detected Model Max Output: {MAX_TOKENS:,} tokens")
    print(f"Average Input Row Size: {avg_input_tokens:.0f} tokens")
    print(f"Maximum Input Rows: {max_rows_by_input:.0f} tokens")
    print(f"Theoretical Max Rows: {theoretical_chunk}")
    print(f"-> Selected dynamic chunk size (Hard Capped): {dynamic_chunk_size} rows")
    print(f"-------------------------------------------")
    
    return dynamic_chunk_size

# ==========================================
# CORE API LOGIC
# ==========================================
def label_chunk(chunk_df: pd.DataFrame, max_retries: int = 5):
    chunk_work = chunk_df.copy()
    chunk_work["row_id"] = chunk_work.index

    csv_input = chunk_work[["row_id", "text", "label"]].to_csv(index=False)
    prompt = build_prompt(csv_input)
    backoff = 4

    print(f"\n[DEBUG] Sending chunk with {len(chunk_work)} rows (Indices {chunk_work.index.min()} to {chunk_work.index.max()})...")

    for attempt in range(1, max_retries + 1):
        try:
            # Assume rate_limited_wait() is defined elsewhere in your code
            # rate_limited_wait() 
            
            response = model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    temperature=0.6,
                    top_p=0.7,
                    max_output_tokens=MAX_TOKENS,
                )
            )

            try:
                raw = response.text
                # Print a preview of the raw text so we know the model responded
                preview = raw[:150].replace('\n', '\\n') + " ... " + raw[-100:].replace('\n', '\\n')
                print(f"  [DEBUG] Attempt {attempt} - Raw API Response preview: {preview}")
                print(f"  [DEBUG] Attempt {attempt} - Raw length: {len(raw)} characters.")
            except ValueError:
                raw = ""
                print(f"  [DEBUG] Attempt {attempt} - Blocked or empty response.")

            result = extract_csv(raw, expected_rows=len(chunk_work))
            
            if result is not None:
                print(f"  [DEBUG] CSV Parsed successfully. Returned {len(result)} rows.")
                return result
            else:
                print(f"  [DEBUG] extract_csv returned None. Retrying...")

        except Exception as exc:
            print(f"  [DEBUG] API Exception on attempt {attempt}: {exc}")
            if "429" in str(exc) or "quota" in str(exc).lower():
                time.sleep(backoff)
                backoff = min(backoff * 2, 120)

    print("  [DEBUG] Chunk failed after max retries. Returning None.")
    return None

def label_dataframe(df: pd.DataFrame, tpm_limit: int = TPM_LIMIT, safe_zone_pct: float = SAFE_ZONE_PERCENT) -> pd.DataFrame:
    chunk_size = calculate_dynamic_chunk_size(df, tpm_limit, safe_zone_pct)
    total = len(df)
    
    if os.path.exists(OUTPUT_CSV): 
        print(f"Found existing {OUTPUT_CSV}. Loading to resume...")
        out = pd.read_csv(OUTPUT_CSV)
        for col in ["gemini_reasoning", "gemini_score", "gemini_rating", "gemini_dominant_signals"]:
            if col not in out.columns:
                out[col] = None
    else:
        out = df.copy().reset_index(drop=True)
        for col in ["gemini_reasoning", "gemini_score", "gemini_rating", "gemini_dominant_signals"]:
            out[col] = None

    # FIX: Explicitly cast columns to the correct data types to prevent FutureWarnings
    out["gemini_reasoning"] = out["gemini_reasoning"].astype(object)
    out["gemini_rating"] = out["gemini_rating"].astype(object)
    out["gemini_dominant_signals"] = out["gemini_dominant_signals"].astype(object)
    out["gemini_score"] = pd.to_numeric(out["gemini_score"], errors='coerce')
            
    unprocessed_mask = out['gemini_score'].isna()
    unprocessed_indices = out[unprocessed_mask].index.tolist()
    
    if not unprocessed_indices:
        return out
        
    n_chunks = (len(unprocessed_indices) + chunk_size - 1) // chunk_size

    with tqdm(total=n_chunks, unit="chunk", colour="cyan") as pbar:
        for i in range(n_chunks):
            start_idx = i * chunk_size
            end_idx = min((i + 1) * chunk_size, len(unprocessed_indices))
            chunk_indices = unprocessed_indices[start_idx:end_idx]
            
            chunk = out.loc[chunk_indices].copy()
            result = label_chunk(chunk)

            if result is not None:
                mapped_count = 0
                for _, row in result.iterrows():
                    pos = int(row["row_id"])
                    if 0 <= pos < total:
                        out.at[pos, "gemini_reasoning"] = row["reasoning"]
                        out.at[pos, "gemini_score"] = row["score"]
                        out.at[pos, "gemini_rating"] = row["rating"]
                        out.at[pos, "gemini_dominant_signals"] = row["dominant_signals"]
                        mapped_count += 1
                print(f"  [DEBUG] Successfully mapped {mapped_count} rows back to dataframe.")
            else:
                print(f"  [DEBUG] Result was None, mapping skipped for this chunk.")

            # Save immediately
            out.to_csv(OUTPUT_CSV, index=False)
            pbar.update(1)
            
            if i < n_chunks - 1:
                time.sleep(6) # Adjust rate limit sleep as needed

    return out

In [455]:
if __name__ == "__main__":
    df = pd.read_csv(INPUT_CSV)
    df_labeled = label_dataframe(df)
    print(df_labeled.head())

--- [DEBUG] Dynamic Chunk Configuration ---
Model Selected: models/gemini-3.1-flash-lite
Detected Model Max Output: 65,536 tokens
Average Input Row Size: 115 tokens
Maximum Input Rows: 522 tokens
Theoretical Max Rows: 204
-> Selected dynamic chunk size (Hard Capped): 204 rows
-------------------------------------------
Found existing /kaggle/working//gemini_labels_output_new.csv. Loading to resume...


  0%|          | 0/118 [00:00<?, ?chunk/s]


[DEBUG] Sending chunk with 204 rows (Indices 4041 to 4244)...
  [DEBUG] Attempt 1 - Raw API Response preview: row_id,reasoning,dominant_signals,score,rating\n4041,"The text uses extremely derogatory language and self-blame, indicating significant internal distr ... ey. This is a moderate, non-crisis level of distress.","reflection, mental health history",5,NEUTRAL
  [DEBUG] Attempt 1 - Raw length: 33236 characters.
  [DEBUG] CSV Parsed successfully. Returned 204 rows.
  [DEBUG] Successfully mapped 204 rows back to dataframe.


  1%|          | 1/118 [00:36<1:11:57, 36.90s/chunk]


[DEBUG] Sending chunk with 204 rows (Indices 4245 to 4448)...
  [DEBUG] Attempt 1 - Raw API Response preview: row_id,reasoning,dominant_signals,score,rating\n4245,"The text is a short, neutral query about a platform. No indicators of mental health distress or n ... success",1,BEST\n4448,"The user is sharing a joke. No signs of distress.","humor, no distress",1,BEST
  [DEBUG] Attempt 1 - Raw length: 27303 characters.
  [DEBUG] CSV Parsed successfully. Returned 204 rows.
  [DEBUG] Successfully mapped 204 rows back to dataframe.


...


[DEBUG] Sending chunk with 204 rows (Indices 27705 to 27908)...
  [DEBUG] Attempt 1 - Raw API Response preview: row_id,reasoning,dominant_signals,score,rating\n27705,"The user expresses feelings of being a 'loser' and mentions being targeted, cheated, and taunted ...  and having suicidal ideation. This is a severe crisis.","suicidal ideation, abuse, crisis",10,WORST
  [DEBUG] Attempt 1 - Raw length: 29587 characters.
  [DEBUG] CSV Parsed successfully. Returned 204 rows.
  [DEBUG] Successfully mapped 204 rows back to dataframe.


 99%|█████████▉| 117/118 [1:30:08<01:02, 62.37s/chunk]


[DEBUG] Sending chunk with 68 rows (Indices 27909 to 27976)...
  [DEBUG] Attempt 1 - Raw API Response preview: 27909,"The text expresses frustration and interpersonal conflict but lacks clinical indicators of severe mental health issues. It reflects a difficult ... t is a casual, confused social inquiry. No signs of distress.","neutral, social inquiry",2,VERY_GOOD
  [DEBUG] Attempt 1 - Raw length: 12293 characters.
  [DEBUG] extract_csv returned None. Retrying...
  [DEBUG] Attempt 2 - Raw API Response preview: row_id,reasoning,dominant_signals,score,rating\n27909,"The text is incoherent and aggressive, showing signs of emotional volatility and personal distre ... about an internet interaction. No signs of mental health distress.","neutral, confusion",2,VERY_GOOD
  [DEBUG] Attempt 2 - Raw length: 13113 characters.
  [DEBUG] CSV Parsed successfully. Returned 68 rows.
  [DEBUG] Successfully mapped 68 rows back to dataframe.


100%|██████████| 118/118 [1:30:44<00:00, 46.14s/chunk]

                                                text  label  \
0  dear american teens question dutch person hear...      0   
1  nothing look forward lifei dont many reasons k...      1   
2  music recommendations im looking expand playli...      0   
3  im done trying feel betterthe reason im still ...      1   
4  worried  year old girl subject domestic physic...      1   

                                    gemini_reasoning  gemini_score  \
0  The text is a casual, incoherent inquiry about...           2.0   
1  The user explicitly mentions wanting to hang t...          10.0   
2  The user is seeking music recommendations and ...           1.0   
3  The user expresses severe hopelessness, suicid...           9.0   
4  The user describes a history of severe domesti...           9.0   

   gemini_rating                            gemini_dominant_signals  
0      VERY_GOOD      incoherent inquiry, no distress, neutral tone  
1          WORST  explicit suicidal ideation, hopelessness, 